In [4]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** 12
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

Daniel Lei
Turki 
Ruoxi
Grace

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

In [5]:

tickers_df = pd.read_csv('Tickers_Example.csv')

start_date = '2024-10-01'
end_date = '2025-09-30'

# list of all the tickers in the CSV file

ticker_list = tickers_df.iloc[:, 0].tolist()

In [6]:
new_ticker_list = []
for ticker in ticker_list:
    new_ticker_list.append(ticker.replace(' ', '').upper())

ticker_list = new_ticker_list.copy()
ticker_list

['ABBV',
 'ABT',
 'ACN',
 'AGN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BK',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CELG',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MON',
 'MRK',
 'PEP',
 'PFE',
 'RY.TO',
 'PM',
 'PYPL',
 'QCOM',
 'RTN',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [7]:
visited = []
new_ticker_list = []
for ticker in ticker_list:
    # find if the actual stock name (without the .TO or any market) already exists in the ticker list
    temp_ticker = ticker.split('.')
    temp_ticker = temp_ticker[0]
    if temp_ticker in visited:
        continue
    new_ticker_list.append(ticker)
    visited.append(temp_ticker)

ticker_list = new_ticker_list.copy()

In [8]:
ticker_data = yf.download(ticker_list, start=start_date, end=end_date, auto_adjust=True)
ticker_data

[**********************72%**********             ]  28 of 39 completed$RTN: possibly delisted; no timezone found
[**********************74%***********            ]  29 of 39 completed$AGN: possibly delisted; no timezone found
[**********************77%************           ]  30 of 39 completed$BK: possibly delisted; no price data found  (1d 2024-10-01 -> 2025-09-30) (Yahoo error = "Data doesn't exist for startDate = 1727755200, endDate = 1759204800")
[**********************79%*************          ]  31 of 39 completed$CELG: possibly delisted; no timezone found
[**********************95%*********************  ]  37 of 39 completed$MON: possibly delisted; no timezone found
[*********************100%***********************]  39 of 39 completed

5 Failed downloads:
['RTN', 'AGN', 'CELG', 'MON']: possibly delisted; no timezone found
['BK']: possibly delisted; no price data found  (1d 2024-10-01 -> 2025-09-30) (Yahoo error = "Data doesn't exist for startDate = 1727755200, endDate = 17592

Price      Adj Close                        Close                              \
Ticker           AGN  BK CELG MON RTN        ABBV         ABT         ACN AGN   
Date                                                                            
2024-10-01       NaN NaN  NaN NaN NaN  183.311874  108.235298  335.945465 NaN   
2024-10-02       NaN NaN  NaN NaN NaN  183.181595  108.311546  340.059082 NaN   
2024-10-03       NaN NaN  NaN NaN NaN  181.906509  106.967659  344.783630 NaN   
2024-10-04       NaN NaN  NaN NaN NaN  180.826904  107.358429  345.738068 NaN   
2024-10-07       NaN NaN  NaN NaN NaN  180.194016  108.044678  340.545898 NaN   
...              ...  ..  ...  ..  ..         ...         ...         ...  ..   
2025-09-23       NaN NaN  NaN NaN NaN  214.594147  132.007385  228.908813 NaN   
2025-09-24       NaN NaN  NaN NaN NaN  212.068481  130.064529  232.388611 NaN   
2025-09-25       NaN NaN  NaN NaN NaN  210.670715  129.501114  226.051086 NaN   
2025-09-26       NaN NaN  NaN NaN NaN  212.666168  129.734253  232.281693 NaN   
2025-09-29       NaN NaN  NaN NaN NaN  215.124344  129.306839  240.086945 NaN   

Price                  ... Volume                                   \
Ticker            AIG  ...    RTN      RY.TO    SHOP.TO       T.TO   
Date                   ...                                           
2024-10-01  69.465912  ...    NaN  3651900.0  2157400.0  1938900.0   
2024-10-02  69.207993  ...    NaN  1914600.0  1365900.0  3221700.0   
2024-10-03  69.102905  ...    NaN  2695800.0  2480500.0  3113300.0   
2024-10-04  72.646896  ...    NaN  4613500.0  3274600.0  2592100.0   
2024-10-07  70.344742  ...    NaN  5066700.0  3953500.0  3770500.0   
...               ...  ...    ...        ...        ...        ...   
2025-09-23  74.856033  ...    NaN  1942700.0  1699600.0  3666100.0   
2025-09-24  76.027184  ...    NaN  1743800.0  1302300.0  3533400.0   
2025-09-25  75.246414  ...    NaN  3413500.0  1449800.0  2577600.0   
2025-09-26  76.105263  ...    NaN  2145400.0  2020100.0  2884100.0   
2025-09-29  76.027184  ...    NaN  2485200.0  2133700.0  3283900.0   

Price                                                                           
Ticker           TD.TO        TXN         UNH        UNP        UPS        USB  
Date                                                                            
2024-10-01  11345000.0  9039900.0   2648400.0  1925000.0  3658500.0  6272800.0  
2024-10-02  16069600.0  4389400.0   3667000.0  2568300.0  2258400.0  6114300.0  
2024-10-03   6156500.0  3167000.0   2645800.0  1990200.0  3635000.0  4513200.0  
2024-10-04  10912000.0  4092900.0   2462600.0  2137900.0  3270500.0  5438000.0  
2024-10-07  16108000.0  2772400.0   2591100.0  1684100.0  1955600.0  5108400.0  
...                ...        ...         ...        ...        ...        ...  
2025-09-23   3468500.0  5900800.0  12504100.0  5183000.0  7486300.0  9970900.0  
2025-09-24   4095000.0  5905100.0   9016300.0  5925900.0  6988400.0  6621100.0  
2025-09-25   4543100.0  4042300.0   8104900.0  6034400.0  9021100.0  7199000.0  
2025-09-26   2800300.0  6206300.0   6774700.0  4940900.0  6845800.0  8309500.0  
2025-09-29   9934500.0  5090700.0   6878900.0  5097300.0  7275300.0  9379500.0  

[255 rows x 200 columns]

In [9]:
temp_list = ticker_data['Close']
ticker_list = temp_list.columns.tolist()
ticker_list

['ABBV',
 'ABT',
 'ACN',
 'AGN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BK',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CELG',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MON',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RTN',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [10]:
# Go through every month, remove any month for a ticker where their trading days are < 18
trading_days = ticker_data.index
filtered_ticker_data = ticker_data.copy()
for ticker in ticker_list:
    
    # gets the number of days in each month with values, basically meaning the number of trading days
    days_each_month = filtered_ticker_data['Close'][ticker].resample('ME').count()
    days_each_month.index = days_each_month.index.strftime('%Y-%m')

    # go through every day for this ticker, and set every day that's in a month with < 18 trading days to None
    for day in trading_days:
        #print(days_each_month.loc[day.strftime('%Y-%m')])
        if days_each_month.loc[day.strftime('%Y-%m')] < 18:
            # setting these values to None so that they are skipped when calculating numbers such as .mean() or .pct_change()
            filtered_ticker_data.loc[day, ('Close', ticker)] = None 
            filtered_ticker_data.loc[day, ('Volume', ticker)] = None

In [11]:
# Remove every stock with average trade volume < 5000 shares
volumes = filtered_ticker_data['Volume'].copy()
mean_volumes = volumes.mean()

min_volume = 5000

# The rest of the code will only reference stocks from filtered_tickers, so any stocks with volume < 5000 are removed
filtered_tickers = mean_volumes[mean_volumes >= min_volume].index.tolist()

# These are the remaining valid tickers
filtered_tickers

['ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [12]:
# Remove all foreign stocks
fil = filtered_tickers.copy()

# Goes through all the tickers in the current list of tickers, gets their country, and if it's not US or Canada, it gets removed 
for ticker in fil:
    tick = yf.Ticker(ticker)
    info = tick.get_info()

    # NEW CODE in this section
    type_stock = info['quoteType']
    if type_stock != 'EQUITY':
        print(f'Ticker removed for not being a stock: {ticker} {type_stock}')
        filtered_tickers.remove(ticker)
    # End of new code
    
    else:
        country = info['country']
        #print(country)
        if country == 'United States' or country == 'Canada':
            continue
        
        print(f'Stock removed for not being a US or Canadian company: {ticker}')
        filtered_tickers.remove(ticker)

# These are the remaining valid tickers
filtered_tickers

Stock removed for not being a US or Canadian company: ACN


['ABBV',
 'ABT',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

After removing all the unwanted stocks, get data for each of the remaining stocks that will help us determine if they are stable and optimal for us to add to our portfolio:
Standard deviation
Tells us how volatile/risky the stock is, and we will look for stocks that have lower standard deviations (close to 0) to reduce the volatility and risk of our portfolio
Beta to the S&P 500
Tells us how closely the stock moves with the market. We want to look for stocks that have lower betas (close to 0 in absolute value) because it will mean that the stock's movements aren't very correlated with the market, and don't particularly go up or down when the market moves
Expected Monthly Returns
Tells us what percentage return we should expect to see per month. We want to look for stocks that have lower expected monthly returns (close to 0 in absolute value) because we want a portfolio that gives as close to 0% returns as possible

In [13]:
# Monthly Returns

close = filtered_ticker_data['Close'].copy()
close = close[filtered_tickers]
monthly_returns=close.resample('ME').first().pct_change()
monthly_returns.drop(index=monthly_returns.index[0], inplace=True)

# Expected Monthly Returns
monthly_returns_list = abs(monthly_returns.mean()) # do absolute value because we want it as close to 0 as possible
monthly_returns_list

Ticker
ABBV       0.011840
ABT        0.016790
AIG        0.013283
AMZN       0.020389
AXP        0.023071
BA         0.041401
BAC        0.027318
BB.TO      0.057476
BIIB       0.026341
BLK        0.018428
BMY        0.003241
C          0.045388
CAT        0.009555
CL         0.014211
KO         0.000014
LLY        0.013606
LMT        0.022079
MO         0.031491
MRK        0.022317
PEP        0.007982
PFE        0.005387
PM         0.035815
PYPL       0.005685
QCOM       0.000494
RY.TO      0.020285
SHOP.TO    0.063653
T.TO       0.005994
TD.TO      0.022355
TXN        0.006623
UNH        0.038822
UNP        0.005544
UPS        0.030433
USB        0.013557
dtype: float64

## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

| Signal | Why it points toward higher return | Weight |
|---|---|---|
| **Beta** (Chapter 15) | In a rising market, a beta-1.5 stock tends to move about 1.5× the market. CAPM: more market risk earns more expected return. | 40% |
| **12-1 month momentum** | Past-year winners (skipping the latest month, which tends to bounce back) have tended to keep winning (Jegadeesh & Titman, 1993). Research evidence, not from our slides. | 40% |
| **Revenue growth** (Yahoo) | Our client wants *growth*; fast-growing sales support a company's story. Yahoo's data is patchy, so it gets less weight and **a missing value counts as neutral**. | 20% |

**Selection:** top 12 by score, at most 4 per sector. **Weighting:** proportional to score (shifted so the lowest of the 12 is still positive),
then the 15% / 40% caps are applied and any excess is spread over the uncapped stocks.
**What we deliberately did *not* use:** dividends — the competition ignores dividends, so they earn nothing in this contest.

### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.
Client A aims for the maximum return and is willing to take high risks. As expected returns cannot be observed, I have measured each stock against three factors and aggregated them using their z-scores (their distance from the mean of the entire set). Beta will account for 40%: stock with beta greater than 1 will move even more upwards than the movement of the market, and hence, the theory suggests that investors get higher compensation for market risk according to CAPM. Momentum over 12-1 month period will get 40%. 20% for revenue growth that corresponds to the growth strategy, although it is given less weight since the data for Yahoo is unreliable and the absence of a figure means a neutral score. I have purchased the top 12 based on their scores, where each sector is allowed a maximum of 4 stocks so that no single sector will be dominant. This was done intentionally since the contest gives recognition solely based on the best performing stock over five days, and with fewer stocks, there is greater variation in performance. Higher scores receive larger allocations with the stock limit of 15% and the sector limit of 40%. 

Client A is likely high-scoring (higher beta, growth, strong past year): so stocks like 
AMZN, SHOP.TO, QCOM, LLY, PYPL, AXP, CAT, BA, plus possibly C, BAC, TXN and ACN. BB.TO is very volatile and could score high or low depending on its momentum. The 4-per-sector limit will push the list to spread across tech, financials, industrials and healthcare.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.
If the market falls between Nov 30 and Dec 4, high-beta stocks will fall more than the market, so the beta tilt works against us. Momentum can reverse suddenly, and over only five days luck matters more than any signal. Up to 40% of the portfolio can sit in one sector, so a sector-specific shock would hurt. US stocks also lose value in CAD if the Canadian dollar strengthens.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.